# Satellite Change Detection Dataset Audit Notebook

This notebook scans your local dataset root and produces a structured inventory for common remote-sensing change-detection datasets:

- **Binary building change:** LEVIR-CD, WHU-CD
- **General / seasonal change:** DSIFN-CD, CDD
- **Semantic change:** SECOND, HRSCD
- **Disaster-related change:** xBD / xView2
- **Viewpoint robustness:** S2Looking

It checks available directories, file counts, file formats, approximate dataset size, common train/val/test splits, and likely image/label folders.

Default root path: `D:\Datasets\Satillate`

> Note: The folder name is kept exactly as you provided: **Satillate**. If your actual folder is `Satellite`, update `ROOT_DIR` in Cell 2.

In [1]:
# Cell 1: Imports and display settings
from pathlib import Path
from collections import Counter, defaultdict
import os
import json
import csv
import re
import math
import time
from datetime import datetime

import pandas as pd

try:
    from PIL import Image
    PIL_AVAILABLE = True
except Exception:
    PIL_AVAILABLE = False

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 80)
pd.set_option('display.max_colwidth', 160)
print('Pandas:', pd.__version__)
print('PIL available:', PIL_AVAILABLE)

Pandas: 2.3.3
PIL available: True


In [2]:
# Cell 2: Configure your dataset root
# Keep this exactly as your Windows path. Change only if your folder is named differently.
ROOT_DIR = Path(r'D:\Datasets\Satillate')

# Output audit folder will be created inside ROOT_DIR if it exists, otherwise beside this notebook.
OUTPUT_DIR = ROOT_DIR / '_dataset_audit_reports' if ROOT_DIR.exists() else Path.cwd() / '_dataset_audit_reports'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Root:', ROOT_DIR)
print('Root exists:', ROOT_DIR.exists())
print('Audit output:', OUTPUT_DIR.resolve())

if not ROOT_DIR.exists():
    print('\nWARNING: Root folder does not exist. Please check spelling/path.')
    print('Common issue: you wrote Satillate, but the folder may be Satellite.')

Root: D:\Datasets\Satillate
Root exists: True
Audit output: D:\Datasets\Satillate\_dataset_audit_reports


In [3]:
# Cell 3: Dataset definitions and filename patterns
DATASET_GROUPS = {
    'Binary building change': ['LEVIR-CD', 'WHU-CD'],
    'General / seasonal change': ['DSIFN-CD', 'CDD'],
    'Semantic change': ['SECOND', 'HRSCD'],
    'Disaster-related change': ['xBD', 'xView2'],
    'Viewpoint robustness': ['S2Looking'],
}

# Flexible aliases for detecting folders/files.
DATASET_ALIASES = {
    'LEVIR-CD': ['levir', 'levir-cd', 'levir_cd', 'levir cd'],
    'WHU-CD': ['whu', 'whu-cd', 'whu_cd', 'whu cd', 'whu building'],
    'DSIFN-CD': ['dsifn', 'dsifn-cd', 'dsifn_cd', 'dsifn cd'],
    'CDD': ['cdd', 'change detection dataset'],
    'SECOND': ['second', 'semantic change detection'],
    'HRSCD': ['hrscd', 'high resolution semantic change'],
    'xBD': ['xbd', 'xview2', 'xview 2', 'xview-2'],
    'xView2': ['xview2', 'xview 2', 'xview-2', 'xbd'],
    'S2Looking': ['s2looking', 's2 looking', 's2-looking'],
}

IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.tif', '.tiff', '.bmp', '.webp', '.jp2'}
LABEL_EXTENSIONS = {'.png', '.tif', '.tiff', '.json', '.geojson', '.txt', '.csv', '.xml'}
ARCHIVE_EXTENSIONS = {'.zip', '.rar', '.7z', '.tar', '.gz', '.tgz'}
TABLE_EXTENSIONS = {'.csv', '.xlsx', '.xls', '.json', '.geojson', '.txt'}

SPLIT_KEYWORDS = ['train', 'val', 'valid', 'validation', 'test']
ROLE_KEYWORDS = {
    'pre_image': ['a', 'before', 'pre', 'pre-disaster', 'pre_event', 't1', 'time1', 'image1', 'im1'],
    'post_image': ['b', 'after', 'post', 'post-disaster', 'post_event', 't2', 'time2', 'image2', 'im2'],
    'label': ['label', 'labels', 'mask', 'masks', 'gt', 'ground_truth', 'target', 'change', 'annotation', 'ann'],
    'semantic_label': ['semantic', 'landcover', 'land_cover', 'class', 'segmentation'],
    'metadata': ['metadata', 'csv', 'json', 'list', 'splits']
}

print('Expected datasets:')
for group, ds in DATASET_GROUPS.items():
    print(f'- {group}: {", ".join(ds)}')

Expected datasets:
- Binary building change: LEVIR-CD, WHU-CD
- General / seasonal change: DSIFN-CD, CDD
- Semantic change: SECOND, HRSCD
- Disaster-related change: xBD, xView2
- Viewpoint robustness: S2Looking


In [4]:
# Cell 4: Helper functions

def normalize_text(s):
    return re.sub(r'[^a-z0-9]+', ' ', str(s).lower()).strip()


def bytes_to_human(n):
    if n is None:
        return ''
    n = float(n)
    units = ['B', 'KB', 'MB', 'GB', 'TB']
    idx = 0
    while n >= 1024 and idx < len(units) - 1:
        n /= 1024
        idx += 1
    return f'{n:.2f} {units[idx]}'


def safe_iterdir(path):
    try:
        yield from path.iterdir()
    except Exception as e:
        return


def safe_stat(path):
    try:
        return path.stat()
    except Exception:
        return None


def walk_limited(root, max_files=None):
    count = 0
    for dirpath, dirnames, filenames in os.walk(root):
        dirpath = Path(dirpath)
        for fname in filenames:
            yield dirpath / fname
            count += 1
            if max_files and count >= max_files:
                return


def get_top_level_entries(root):
    if not root.exists():
        return []
    rows = []
    for p in safe_iterdir(root):
        st = safe_stat(p)
        rows.append({
            'name': p.name,
            'path': str(p),
            'type': 'directory' if p.is_dir() else 'file',
            'size_bytes': st.st_size if st else None,
            'modified': datetime.fromtimestamp(st.st_mtime).isoformat(timespec='seconds') if st else None,
        })
    return sorted(rows, key=lambda x: (x['type'] != 'directory', x['name'].lower()))


def detect_dataset_name(path):
    text = normalize_text(path.name + ' ' + str(path))
    hits = []
    for ds, aliases in DATASET_ALIASES.items():
        score = 0
        for alias in aliases:
            a = normalize_text(alias)
            if a and a in text:
                score += len(a)
        if score:
            hits.append((score, ds))
    if not hits:
        return None
    hits.sort(reverse=True)
    return hits[0][1]


def infer_split_from_path(path):
    parts = [normalize_text(p) for p in path.parts]
    for part in parts:
        for key in SPLIT_KEYWORDS:
            if key == part or key in part.split():
                if key in ['valid', 'validation']:
                    return 'val'
                return key
    return 'unknown'


def infer_role_from_path(path):
    parts_text = normalize_text(' '.join(path.parts))
    ext = path.suffix.lower()
    role_scores = Counter()
    for role, keywords in ROLE_KEYWORDS.items():
        for kw in keywords:
            nkw = normalize_text(kw)
            if nkw and nkw in parts_text:
                role_scores[role] += len(nkw)
    if ext in {'.json', '.geojson', '.csv', '.txt', '.xml'}:
        role_scores['metadata'] += 2
    if not role_scores:
        if ext in IMAGE_EXTENSIONS:
            return 'image_unknown'
        return 'other'
    return role_scores.most_common(1)[0][0]


def image_info(path):
    if not PIL_AVAILABLE or path.suffix.lower() not in IMAGE_EXTENSIONS:
        return None
    try:
        with Image.open(path) as im:
            return {'width': im.width, 'height': im.height, 'mode': im.mode}
    except Exception:
        return None

print('Helpers ready.')

Helpers ready.


In [5]:
# Cell 5: Show top-level folders/files
entries = get_top_level_entries(ROOT_DIR)
top_df = pd.DataFrame(entries)
if len(top_df):
    display(top_df)
else:
    print('No entries found, or root path does not exist.')

top_df.to_csv(OUTPUT_DIR / 'top_level_inventory.csv', index=False)
print('Saved:', OUTPUT_DIR / 'top_level_inventory.csv')

,name,path,type,size_bytes,modified
0,_dataset_audit_reports,D:\Datasets\Satillate\_dataset_audit_reports,directory,0,2026-06-03T10:38:34
1,Building change detection dataset_add,D:\Datasets\Satillate\Building change detection dataset_add,directory,0,2026-06-02T14:39:43
2,DSIFN Train Test,D:\Datasets\Satillate\DSIFN Train Test,directory,0,2026-06-02T14:49:27
3,HRSCD,D:\Datasets\Satillate\HRSCD,directory,0,2026-06-03T09:39:54
4,LEVIR CD,D:\Datasets\Satillate\LEVIR CD,directory,0,2026-06-02T14:21:39
5,S2Looking,D:\Datasets\Satillate\S2Looking,directory,0,2026-06-02T10:21:48
6,SECOND,D:\Datasets\Satillate\SECOND,directory,0,2026-06-02T09:25:19
7,second_dataset,D:\Datasets\Satillate\second_dataset,directory,0,2026-06-03T09:59:40
8,xView2,D:\Datasets\Satillate\xView2,directory,0,2026-06-02T13:54:01


Saved: D:\Datasets\Satillate\_dataset_audit_reports\top_level_inventory.csv


In [6]:
# Cell 6: Locate expected datasets in the root folder
candidate_rows = []

if ROOT_DIR.exists():
    # Check top-level folders and files first
    for p in safe_iterdir(ROOT_DIR):
        ds = detect_dataset_name(p)
        if ds:
            candidate_rows.append({
                'dataset_detected': ds,
                'candidate_path': str(p),
                'type': 'directory' if p.is_dir() else 'file',
                'matched_from': 'top_level',
            })

    # Also search one level deeper for nested/extracted folders
    for p in safe_iterdir(ROOT_DIR):
        if p.is_dir():
            for q in safe_iterdir(p):
                ds = detect_dataset_name(q)
                if ds:
                    candidate_rows.append({
                        'dataset_detected': ds,
                        'candidate_path': str(q),
                        'type': 'directory' if q.is_dir() else 'file',
                        'matched_from': 'one_level_deep',
                    })

candidates_df = pd.DataFrame(candidate_rows).drop_duplicates() if candidate_rows else pd.DataFrame(columns=['dataset_detected','candidate_path','type','matched_from'])

def dataset_group(ds):
    for g, items in DATASET_GROUPS.items():
        if ds in items:
            return g
    return 'Unknown'

if len(candidates_df):
    candidates_df.insert(0, 'group', candidates_df['dataset_detected'].map(dataset_group))
    display(candidates_df)
else:
    print('No expected dataset folders detected automatically.')

# Availability checklist
expected_rows = []
for group, datasets in DATASET_GROUPS.items():
    for ds in datasets:
        detected = ds in set(candidates_df['dataset_detected']) if len(candidates_df) else False
        paths = candidates_df.loc[candidates_df['dataset_detected'] == ds, 'candidate_path'].tolist() if len(candidates_df) else []
        expected_rows.append({'group': group, 'dataset': ds, 'detected': detected, 'candidate_paths': ' | '.join(paths)})

availability_df = pd.DataFrame(expected_rows)
display(availability_df)

candidates_df.to_csv(OUTPUT_DIR / 'detected_dataset_candidates.csv', index=False)
availability_df.to_csv(OUTPUT_DIR / 'dataset_availability_checklist.csv', index=False)
print('Saved candidate and availability reports.')

,group,dataset_detected,candidate_path,type,matched_from
0,General / seasonal change,CDD,D:\Datasets\Satillate\Building change detection dataset_add,directory,top_level
1,General / seasonal change,DSIFN-CD,D:\Datasets\Satillate\DSIFN Train Test,directory,top_level
2,Semantic change,HRSCD,D:\Datasets\Satillate\HRSCD,directory,top_level
3,Binary building change,LEVIR-CD,D:\Datasets\Satillate\LEVIR CD,directory,top_level
4,Viewpoint robustness,S2Looking,D:\Datasets\Satillate\S2Looking,directory,top_level
5,Semantic change,SECOND,D:\Datasets\Satillate\SECOND,directory,top_level
6,Semantic change,SECOND,D:\Datasets\Satillate\second_dataset,directory,top_level
7,Disaster-related change,xView2,D:\Datasets\Satillate\xView2,directory,top_level
8,General / seasonal change,CDD,D:\Datasets\Satillate\Building change detection dataset_add\1. The two-period image data,directory,one_level_deep
9,General / seasonal change,CDD,D:\Datasets\Satillate\Building change detection dataset_add\2. The shape file of the images,directory,one_level_deep


,group,dataset,detected,candidate_paths
0,Binary building change,LEVIR-CD,True,D:\Datasets\Satillate\LEVIR CD | D:\Datasets\Satillate\LEVIR CD\test | D:\Datasets\Satillate\LEVIR CD\train | D:\Datasets\Satillate\LEVIR CD\val
1,Binary building change,WHU-CD,False,
2,General / seasonal change,DSIFN-CD,True,D:\Datasets\Satillate\DSIFN Train Test | D:\Datasets\Satillate\DSIFN Train Test\test | D:\Datasets\Satillate\DSIFN Train Test\train
3,General / seasonal change,CDD,True,D:\Datasets\Satillate\Building change detection dataset_add | D:\Datasets\Satillate\Building change detection dataset_add\1. The two-period image data | D:\...
4,Semantic change,SECOND,True,D:\Datasets\Satillate\SECOND | D:\Datasets\Satillate\second_dataset | D:\Datasets\Satillate\SECOND\test | D:\Datasets\Satillate\SECOND\train | D:\Datasets\S...
5,Semantic change,HRSCD,True,D:\Datasets\Satillate\HRSCD | D:\Datasets\Satillate\HRSCD\images_2012 | D:\Datasets\Satillate\HRSCD\labels_change | D:\Datasets\Satillate\HRSCD\labels_land_...
6,Disaster-related change,xBD,False,
7,Disaster-related change,xView2,True,D:\Datasets\Satillate\xView2 | D:\Datasets\Satillate\xView2\test | D:\Datasets\Satillate\xView2\train
8,Viewpoint robustness,S2Looking,True,D:\Datasets\Satillate\S2Looking | D:\Datasets\Satillate\S2Looking\test | D:\Datasets\Satillate\S2Looking\train | D:\Datasets\Satillate\S2Looking\val


Saved candidate and availability reports.


In [7]:
# Cell 7: Full file inventory scan
# This can take time for very large datasets. Set MAX_FILES_PER_DATASET to None for complete scan.
MAX_FILES_PER_DATASET = None      # Example for quick test: 10000
SAMPLE_IMAGES_PER_DATASET = 50    # Number of images to inspect for dimensions per dataset

inventory_rows = []
summary_rows = []

if len(candidates_df) == 0:
    print('No dataset candidates found. Run Cell 5 and check folder names.')
else:
    for _, row in candidates_df.iterrows():
        ds = row['dataset_detected']
        group = row['group']
        root = Path(row['candidate_path'])
        if not root.exists():
            continue

        print(f'Scanning {ds}: {root}')
        total_size = 0
        file_count = 0
        dir_count = 0
        ext_counter = Counter()
        split_counter = Counter()
        role_counter = Counter()
        image_dim_counter = Counter()
        image_checked = 0
        large_files = []

        if root.is_file():
            files_iter = [root]
        else:
            for _, dirnames, _ in os.walk(root):
                dir_count += len(dirnames)
            files_iter = walk_limited(root, MAX_FILES_PER_DATASET)

        for f in files_iter:
            st = safe_stat(f)
            size = st.st_size if st else 0
            total_size += size
            file_count += 1
            ext = f.suffix.lower() or '[no_ext]'
            ext_counter[ext] += 1
            split = infer_split_from_path(f)
            role = infer_role_from_path(f)
            split_counter[split] += 1
            role_counter[role] += 1

            if len(large_files) < 20 or size > min(x['size_bytes'] for x in large_files):
                large_files.append({'dataset': ds, 'path': str(f), 'size_bytes': size, 'size': bytes_to_human(size)})
                large_files = sorted(large_files, key=lambda x: x['size_bytes'], reverse=True)[:20]

            dims = None
            if image_checked < SAMPLE_IMAGES_PER_DATASET and ext in IMAGE_EXTENSIONS:
                info = image_info(f)
                image_checked += 1
                if info:
                    dims = f"{info['width']}x{info['height']}:{info['mode']}"
                    image_dim_counter[dims] += 1

            inventory_rows.append({
                'group': group,
                'dataset': ds,
                'root': str(root),
                'relative_path': str(f.relative_to(root)) if root.is_dir() else f.name,
                'extension': ext,
                'size_bytes': size,
                'size': bytes_to_human(size),
                'split_inferred': split,
                'role_inferred': role,
                'image_info_sampled': dims,
                'modified': datetime.fromtimestamp(st.st_mtime).isoformat(timespec='seconds') if st else None,
            })

        summary_rows.append({
            'group': group,
            'dataset': ds,
            'root': str(root),
            'file_count_scanned': file_count,
            'dir_count': dir_count,
            'total_size_bytes': total_size,
            'total_size': bytes_to_human(total_size),
            'top_extensions': json.dumps(ext_counter.most_common(15)),
            'split_counts': json.dumps(split_counter.most_common()),
            'role_counts': json.dumps(role_counter.most_common()),
            'sampled_image_dimensions': json.dumps(image_dim_counter.most_common(20)),
            'largest_files': json.dumps(large_files),
        })

inventory_df = pd.DataFrame(inventory_rows)
summary_df = pd.DataFrame(summary_rows)

print('\nScan completed.')
print('Files scanned:', len(inventory_df))
display(summary_df)

inventory_df.to_csv(OUTPUT_DIR / 'full_file_inventory.csv', index=False)
summary_df.to_csv(OUTPUT_DIR / 'dataset_summary.csv', index=False)
print('Saved:', OUTPUT_DIR / 'full_file_inventory.csv')
print('Saved:', OUTPUT_DIR / 'dataset_summary.csv')

Scanning CDD: D:\Datasets\Satillate\Building change detection dataset_add
Scanning DSIFN-CD: D:\Datasets\Satillate\DSIFN Train Test
Scanning HRSCD: D:\Datasets\Satillate\HRSCD


C:\Users\admin\miniconda3\lib\site-packages\PIL\Image.py:3452: DecompressionBombWarning: Image size (100000000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


Scanning LEVIR-CD: D:\Datasets\Satillate\LEVIR CD
Scanning S2Looking: D:\Datasets\Satillate\S2Looking
Scanning SECOND: D:\Datasets\Satillate\SECOND
Scanning SECOND: D:\Datasets\Satillate\second_dataset
Scanning xView2: D:\Datasets\Satillate\xView2
Scanning CDD: D:\Datasets\Satillate\Building change detection dataset_add\1. The two-period image data
Scanning CDD: D:\Datasets\Satillate\Building change detection dataset_add\2. The shape file of the images
Scanning DSIFN-CD: D:\Datasets\Satillate\DSIFN Train Test\test
Scanning DSIFN-CD: D:\Datasets\Satillate\DSIFN Train Test\train
Scanning HRSCD: D:\Datasets\Satillate\HRSCD\images_2012
Scanning HRSCD: D:\Datasets\Satillate\HRSCD\labels_change
Scanning HRSCD: D:\Datasets\Satillate\HRSCD\labels_land_cover_2006
Scanning HRSCD: D:\Datasets\Satillate\HRSCD\labels_land_cover_2012
Scanning HRSCD: D:\Datasets\Satillate\HRSCD\README.txt
Scanning LEVIR-CD: D:\Datasets\Satillate\LEVIR CD\test
Scanning LEVIR-CD: D:\Datasets\Satillate\LEVIR CD\train
Sc

,group,dataset,root,file_count_scanned,dir_count,total_size_bytes,total_size,top_extensions,split_counts,role_counts,sampled_image_dimensions,largest_files
0,General / seasonal change,CDD,D:\Datasets\Satillate\Building change detection dataset_add,7860,37,7625511175,7.10 GB,"[["".tif"", 7810], ["".xml"", 20], ["".tfw"", 10], ["".ovr"", 10], ["".dbf"", 2], ["".sbn"", 2], ["".sbx"", 2], ["".shp"", 2], ["".shx"", 2]]","[[""train"", 5070], [""test"", 2790]]","[[""label"", 7860]]","[[""512x512:RGB"", 50]]","[{""dataset"": ""CDD"", ""path"": ""D:\\Datasets\\Satillate\\Building change detection dataset_add\\1. The two-period image data\\2016\\whole_image\\train\\image\\..."
1,General / seasonal change,DSIFN-CD,D:\Datasets\Satillate\DSIFN Train Test,47292,8,3646430162,3.40 GB,"[["".png"", 47292]]","[[""train"", 47292]]","[[""pre_image"", 31528], [""label"", 15764]]","[[""256x256:RGB"", 50]]","[{""dataset"": ""DSIFN-CD"", ""path"": ""D:\\Datasets\\Satillate\\DSIFN Train Test\\train\\A\\2191_4.png"", ""size_bytes"": 178409, ""size"": ""174.23 KB""}, {""dataset"": ..."
2,Semantic change,HRSCD,D:\Datasets\Satillate\HRSCD,1165,16,8600896208,8.01 GB,"[["".tif"", 1164], ["".txt"", 1]]","[[""unknown"", 1165]]","[[""label"", 873], [""pre_image"", 291], [""metadata"", 1]]","[[""10000x10000:RGB"", 50]]","[{""dataset"": ""HRSCD"", ""path"": ""D:\\Datasets\\Satillate\\HRSCD\\images_2012\\2012\\D35\\35-2012-0310-6780-LA93-0M50-E080.tif"", ""size_bytes"": 25196577, ""size""..."
3,Binary building change,LEVIR-CD,D:\Datasets\Satillate\LEVIR CD,1911,12,2472439321,2.30 GB,"[["".png"", 1911]]","[[""train"", 1335], [""test"", 384], [""val"", 192]]","[[""pre_image"", 1274], [""label"", 637]]","[[""1024x1024:RGB"", 50]]","[{""dataset"": ""LEVIR-CD"", ""path"": ""D:\\Datasets\\Satillate\\LEVIR CD\\val\\A\\val_19.png"", ""size_bytes"": 2969577, ""size"": ""2.83 MB""}, {""dataset"": ""LEVIR-CD"",..."
4,Viewpoint robustness,S2Looking,D:\Datasets\Satillate\S2Looking,25000,18,11000991200,10.25 GB,"[["".png"", 25000]]","[[""train"", 17500], [""test"", 5000], [""val"", 2500]]","[[""label"", 15000], [""pre_image"", 5000], [""post_image"", 5000]]","[[""1024x1024:RGB"", 50]]","[{""dataset"": ""S2Looking"", ""path"": ""D:\\Datasets\\Satillate\\S2Looking\\train\\Image1\\3408.png"", ""size_bytes"": 2308478, ""size"": ""2.20 MB""}, {""dataset"": ""S2L..."
5,Semantic change,SECOND,D:\Datasets\Satillate\SECOND,6776,6,1383454690,1.29 GB,"[["".png"", 6776]]","[[""test"", 6776]]","[[""label"", 3388], [""pre_image"", 1694], [""post_image"", 1694]]","[[""512x512:RGB"", 50]]","[{""dataset"": ""SECOND"", ""path"": ""D:\\Datasets\\Satillate\\SECOND\\test\\im2\\05466.png"", ""size_bytes"": 608323, ""size"": ""594.07 KB""}, {""dataset"": ""SECOND"", ""p..."
6,Semantic change,SECOND,D:\Datasets\Satillate\second_dataset,18649,11,3796053393,3.54 GB,"[["".png"", 18648], ["".rar"", 1]]","[[""train"", 11873], [""test"", 6776]]","[[""label"", 9324], [""pre_image"", 4663], [""post_image"", 4662]]","[[""512x512:RGB"", 50]]","[{""dataset"": ""SECOND"", ""path"": ""D:\\Datasets\\Satillate\\second_dataset\\test\\im2\\05466.png"", ""size_bytes"": 608323, ""size"": ""594.07 KB""}, {""dataset"": ""SEC..."
7,Disaster-related change,xView2,D:\Datasets\Satillate\xView2,13062,7,11325707888,10.55 GB,"[["".png"", 7464], ["".json"", 5598]]","[[""train"", 11196], [""test"", 1866]]","[[""post_image"", 6531], [""pre_image"", 6531]]","[[""1024x1024:RGB"", 50]]","[{""dataset"": ""xView2"", ""path"": ""D:\\Datasets\\Satillate\\xView2\\train\\train\\images\\santa-rosa-wildfire_00000246_post_disaster.png"", ""size_bytes"": 264925..."
8,General / seasonal change,CDD,D:\Datasets\Satillate\Building change detection dataset_add\1. The two-period image data,7850,33,7625509845,7.10 GB,"[["".tif"", 7810], ["".xml"", 20], ["".tfw"", 10], ["".ovr"", 10]]","[[""train"", 5065], [""test"", 2785]]","[[""label"", 7850]]","[[""512x512:RGB"", 50]]","[{""dataset"": ""CDD"", ""path"": ""D:\\Datasets\\Satillate\\Building change det

Saved: D:\Datasets\Satillate\_dataset_audit_reports\full_file_inventory.csv
Saved: D:\Datasets\Satillate\_dataset_audit_reports\dataset_summary.csv


In [8]:
# Cell 8: Extension, split, and role summary tables
if len(inventory_df) == 0:
    print('No inventory available. Run Cell 7 first.')
else:
    ext_summary = (inventory_df.groupby(['group', 'dataset', 'extension'])
                   .size().reset_index(name='count')
                   .sort_values(['dataset', 'count'], ascending=[True, False]))
    split_summary = (inventory_df.groupby(['group', 'dataset', 'split_inferred'])
                     .size().reset_index(name='count')
                     .sort_values(['dataset', 'count'], ascending=[True, False]))
    role_summary = (inventory_df.groupby(['group', 'dataset', 'role_inferred'])
                    .size().reset_index(name='count')
                    .sort_values(['dataset', 'count'], ascending=[True, False]))

    print('Extension summary')
    display(ext_summary)
    print('Split summary')
    display(split_summary)
    print('Role summary')
    display(role_summary)

    ext_summary.to_csv(OUTPUT_DIR / 'extension_summary.csv', index=False)
    split_summary.to_csv(OUTPUT_DIR / 'split_summary.csv', index=False)
    role_summary.to_csv(OUTPUT_DIR / 'role_summary.csv', index=False)
    print('Saved summary CSVs.')

Extension summary


,group,dataset,extension,count
10,General / seasonal change,CDD,.tif,15620
11,General / seasonal change,CDD,.xml,40
4,General / seasonal change,CDD,.ovr,20
9,General / seasonal change,CDD,.tfw,20
3,General / seasonal change,CDD,.dbf,4
5,General / seasonal change,CDD,.sbn,4
6,General / seasonal change,CDD,.sbx,4
7,General / seasonal change,CDD,.shp,4
8,General / seasonal change,CDD,.shx,4
12,General / seasonal change,DSIFN-CD,.png,94584


Split summary


,group,dataset,split_inferred,count
6,General / seasonal change,CDD,train,10140
5,General / seasonal change,CDD,test,5580
7,General / seasonal change,DSIFN-CD,train,94584
8,Semantic change,HRSCD,unknown,2330
1,Binary building change,LEVIR-CD,train,2670
0,Binary building change,LEVIR-CD,test,768
2,Binary building change,LEVIR-CD,val,384
12,Viewpoint robustness,S2Looking,train,35000
11,Viewpoint robustness,S2Looking,test,10000
13,Viewpoint robustness,S2Looking,val,5000


Role summary


,group,dataset,role_inferred,count
4,General / seasonal change,CDD,label,15720
6,General / seasonal change,DSIFN-CD,pre_image,63056
5,General / seasonal change,DSIFN-CD,label,31528
7,Semantic change,HRSCD,label,1746
9,Semantic change,HRSCD,pre_image,582
8,Semantic change,HRSCD,metadata,2
1,Binary building change,LEVIR-CD,pre_image,2548
0,Binary building change,LEVIR-CD,label,1274
13,Viewpoint robustness,S2Looking,label,30000
14,Viewpoint robustness,S2Looking,post_image,10000


Saved summary CSVs.


In [9]:
# Cell 9: Detect likely paired image and label folders
# This searches directory names and reports likely A/B/label structures.
folder_rows = []

if len(candidates_df):
    for _, row in candidates_df.iterrows():
        ds = row['dataset_detected']
        group = row['group']
        root = Path(row['candidate_path'])
        if not root.exists() or not root.is_dir():
            continue
        for dirpath, dirnames, filenames in os.walk(root):
            dirpath = Path(dirpath)
            rel = str(dirpath.relative_to(root))
            file_exts = Counter(Path(f).suffix.lower() or '[no_ext]' for f in filenames)
            n_images = sum(c for ext, c in file_exts.items() if ext in IMAGE_EXTENSIONS)
            n_labels = sum(c for ext, c in file_exts.items() if ext in LABEL_EXTENSIONS)
            if filenames or any(k in normalize_text(rel) for k in ['train', 'val', 'test', 'label', 'mask', 'gt', 'a', 'b', 'pre', 'post', 'images']):
                folder_rows.append({
                    'group': group,
                    'dataset': ds,
                    'root': str(root),
                    'relative_folder': rel,
                    'num_files_direct': len(filenames),
                    'num_image_files_direct': n_images,
                    'num_label_like_files_direct': n_labels,
                    'top_extensions_direct': json.dumps(file_exts.most_common(10)),
                    'split_inferred': infer_split_from_path(dirpath),
                    'role_inferred': infer_role_from_path(dirpath),
                })

folder_df = pd.DataFrame(folder_rows)
if len(folder_df):
    folder_df = folder_df.sort_values(['dataset', 'split_inferred', 'role_inferred', 'relative_folder'])
    display(folder_df.head(300))
else:
    print('No folder structure detected.')

folder_df.to_csv(OUTPUT_DIR / 'folder_structure_analysis.csv', index=False)
print('Saved:', OUTPUT_DIR / 'folder_structure_analysis.csv')

,group,dataset,root,relative_folder,num_files_direct,num_image_files_direct,num_label_like_files_direct,top_extensions_direct,split_inferred,role_inferred
3,General / seasonal change,CDD,D:\Datasets\Satillate\Building change detection dataset_add,1. The two-period image data\2012\splited_images\test,0,0,0,[],test,label
4,General / seasonal change,CDD,D:\Datasets\Satillate\Building change detection dataset_add,1. The two-period image data\2012\splited_images\test\image,690,690,690,"[["".tif"", 690]]",test,label
5,General / seasonal change,CDD,D:\Datasets\Satillate\Building change detection dataset_add,1. The two-period image data\2012\splited_images\test\label,690,690,690,"[["".tif"", 690]]",test,label
10,General / seasonal change,CDD,D:\Datasets\Satillate\Building change detection dataset_add,1. The two-period image data\2012\whole_image\test,0,0,0,[],test,label
11,General / seasonal change,CDD,D:\Datasets\Satillate\Building change detection dataset_add,1. The two-period image data\2012\whole_image\test\image,5,1,3,"[["".xml"", 2], ["".tfw"", 1], ["".tif"", 1], ["".ovr"", 1]]",test,label
...,...,...,...,...,...,...,...,...,...,...
112,Disaster-related change,xView2,D:\Datasets\Satillate\xView2,train,0,0,0,[],train,pre_image
203,Disaster-related change,xView2,D:\Datasets\Satillate\xView2\train,train,0,0,0,[],train,pre_image
204,Disaster-related change,xView2,D:\Datasets\Satillate\xView2\train,train\images,5598,5598,5598,"[["".png"", 5598]]",train,pre_image
113,Disaster-related change,xView2,D:\Datasets\Satillate\xView2,train\train,0,0,0,[],train,pre_image


Saved: D:\Datasets\Satillate\_dataset_audit_reports\folder_structure_analysis.csv


In [10]:
# Cell 10: Pairing checks for common A/B/label image structures
# This tries to identify whether pre/post/label files have matching basenames.
pairing_rows = []

if len(inventory_df):
    for (ds, root), sub in inventory_df.groupby(['dataset', 'root']):
        root_path = Path(root)
        image_like = sub[sub['extension'].isin(IMAGE_EXTENSIONS)].copy()
        if len(image_like) == 0:
            continue

        # Divide files by inferred role.
        pre = image_like[image_like['role_inferred'].isin(['pre_image'])]
        post = image_like[image_like['role_inferred'].isin(['post_image'])]
        label = image_like[image_like['role_inferred'].isin(['label', 'semantic_label'])]

        def basename_set(df):
            return set(Path(x).stem for x in df['relative_path'].tolist())

        pre_set, post_set, label_set = basename_set(pre), basename_set(post), basename_set(label)
        paired_ab = len(pre_set & post_set) if pre_set and post_set else 0
        paired_all = len(pre_set & post_set & label_set) if pre_set and post_set and label_set else 0

        pairing_rows.append({
            'dataset': ds,
            'root': root,
            'image_files': len(image_like),
            'pre_like_images': len(pre),
            'post_like_images': len(post),
            'label_like_images': len(label),
            'matching_pre_post_basenames': paired_ab,
            'matching_pre_post_label_basenames': paired_all,
            'pairing_comment': 'Good common basename pairing detected' if paired_all > 0 else ('Pre/post pairing detected' if paired_ab > 0 else 'No common basename pairing detected by simple heuristic'),
        })

pairing_df = pd.DataFrame(pairing_rows)
if len(pairing_df):
    display(pairing_df)
else:
    print('No pairing analysis available.')

pairing_df.to_csv(OUTPUT_DIR / 'pairing_check.csv', index=False)
print('Saved:', OUTPUT_DIR / 'pairing_check.csv')

,dataset,root,image_files,pre_like_images,post_like_images,label_like_images,matching_pre_post_basenames,matching_pre_post_label_basenames,pairing_comment
0,CDD,D:\Datasets\Satillate\Building change detection dataset_add,7810,0,0,7810,0,0,No common basename pairing detected by simple heuristic
1,CDD,D:\Datasets\Satillate\Building change detection dataset_add\1. The two-period image data,7810,0,0,7810,0,0,No common basename pairing detected by simple heuristic
2,DSIFN-CD,D:\Datasets\Satillate\DSIFN Train Test,47292,31528,0,15764,0,0,No common basename pairing detected by simple heuristic
3,DSIFN-CD,D:\Datasets\Satillate\DSIFN Train Test\test,9459,6306,0,3153,0,0,No common basename pairing detected by simple heuristic
4,DSIFN-CD,D:\Datasets\Satillate\DSIFN Train Test\train,37833,25222,0,12611,0,0,No common basename pairing detected by simple heuristic
5,HRSCD,D:\Datasets\Satillate\HRSCD,1164,291,0,873,0,0,No common basename pairing detected by simple heuristic
6,HRSCD,D:\Datasets\Satillate\HRSCD\images_2012,291,291,0,0,0,0,No common basename pairing detected by simple heuristic
7,HRSCD,D:\Datasets\Satillate\HRSCD\labels_change,291,0,0,291,0,0,No common basename pairing detected by simple heuristic
8,HRSCD,D:\Datasets\Satillate\HRSCD\labels_land_cover_2006,291,0,0,291,0,0,No common basename pairing detected by simple heuristic
9,HRSCD,D:\Datasets\Satillate\HRSCD\labels_land_cover_2012,291,0,0,291,0,0,No common basename pairing detected by simple heuristic


Saved: D:\Datasets\Satillate\_dataset_audit_reports\pairing_check.csv


In [11]:
# Cell 11: Dataset-specific sanity checks and recommendations
recommendations = []

expected = availability_df.copy() if 'availability_df' in globals() else pd.DataFrame()
if len(expected):
    for _, r in expected.iterrows():
        if not r['detected']:
            recommendations.append({
                'dataset': r['dataset'],
                'status': 'Missing / not detected',
                'recommendation': 'Check whether the dataset is downloaded, extracted, or named with a recognizable alias.'
            })

if 'summary_df' in globals() and len(summary_df):
    for _, r in summary_df.iterrows():
        ds = r['dataset']
        file_count = int(r['file_count_scanned'])
        ext_text = r.get('top_extensions', '')
        role_text = r.get('role_counts', '')
        recs = []
        if file_count == 0:
            recs.append('Folder detected but no files scanned.')
        if any(ext in ext_text for ext in ['.zip', '.rar', '.7z']) and file_count < 10:
            recs.append('Dataset may still be compressed; extract archives before training.')
        if 'label' not in role_text and 'mask' not in role_text and ds not in ['xBD', 'xView2']:
            recs.append('No label/mask folder detected by heuristic; verify ground-truth path manually.')
        if ds in ['xBD', 'xView2'] and '.json' not in ext_text and '.geojson' not in ext_text:
            recs.append('xBD/xView2 usually needs JSON building/damage annotations; verify annotation files.')
        if ds in ['SECOND', 'HRSCD'] and 'semantic' not in role_text:
            recs.append('Semantic labels not clearly detected; verify land-cover/class annotation folders.')
        if ds == 'S2Looking' and not any(k in role_text.lower() for k in ['pre_image', 'post_image', 'label']):
            recs.append('Check side-looking A/B/label folder naming manually.')
        if not recs:
            recs.append('Looks usable from automatic checks; verify sample image/label alignment visually before training.')
        recommendations.append({'dataset': ds, 'status': 'Detected', 'recommendation': ' '.join(recs)})

recommendations_df = pd.DataFrame(recommendations).drop_duplicates()
display(recommendations_df)
recommendations_df.to_csv(OUTPUT_DIR / 'dataset_recommendations.csv', index=False)
print('Saved:', OUTPUT_DIR / 'dataset_recommendations.csv')

,dataset,status,recommendation
0,WHU-CD,Missing / not detected,"Check whether the dataset is downloaded, extracted, or named with a recognizable alias."
1,xBD,Missing / not detected,"Check whether the dataset is downloaded, extracted, or named with a recognizable alias."
2,CDD,Detected,Looks usable from automatic checks; verify sample image/label alignment visually before training.
3,DSIFN-CD,Detected,Looks usable from automatic checks; verify sample image/label alignment visually before training.
4,HRSCD,Detected,Semantic labels not clearly detected; verify land-cover/class annotation folders.
5,LEVIR-CD,Detected,Looks usable from automatic checks; verify sample image/label alignment visually before training.
6,S2Looking,Detected,Looks usable from automatic checks; verify sample image/label alignment visually before training.
7,SECOND,Detected,Semantic labels not clearly detected; verify land-cover/class annotation folders.
9,xView2,Detected,Looks usable from automatic checks; verify sample image/label alignment visually before training.
14,HRSCD,Detected,No label/mask folder detected by heuristic; verify ground-truth path manually. Semantic labels not clearly detected; verify land-cover/class annotation fold...


Saved: D:\Datasets\Satillate\_dataset_audit_reports\dataset_recommendations.csv


In [12]:
# Cell 12: Optional visual sample viewer
# Change DATASET_TO_VIEW to one detected dataset name, e.g. 'LEVIR-CD', 'WHU-CD', 'DSIFN-CD', 'CDD', 'SECOND', 'HRSCD', 'xBD', 'S2Looking'.
DATASET_TO_VIEW = None
NUM_SAMPLES = 6

if DATASET_TO_VIEW is None:
    print('Set DATASET_TO_VIEW to a dataset name to display samples.')
elif not PIL_AVAILABLE:
    print('PIL is not available, cannot display image samples.')
elif 'inventory_df' not in globals() or len(inventory_df) == 0:
    print('Run Cell 7 first.')
else:
    import matplotlib.pyplot as plt
    sub = inventory_df[(inventory_df['dataset'] == DATASET_TO_VIEW) & (inventory_df['extension'].isin(IMAGE_EXTENSIONS))].head(NUM_SAMPLES)
    if len(sub) == 0:
        print('No image files found for', DATASET_TO_VIEW)
    else:
        for _, r in sub.iterrows():
            f = Path(r['root']) / r['relative_path']
            try:
                im = Image.open(f)
                plt.figure(figsize=(6, 6))
                plt.imshow(im)
                plt.axis('off')
                plt.title(f"{DATASET_TO_VIEW} | {r['role_inferred']} | {r['relative_path']}")
                plt.show()
            except Exception as e:
                print('Could not open:', f, e)

Set DATASET_TO_VIEW to a dataset name to display samples.


In [13]:
# Cell 13: Export a compact JSON audit report
report = {
    'created_at': datetime.now().isoformat(timespec='seconds'),
    'root': str(ROOT_DIR),
    'root_exists': ROOT_DIR.exists(),
    'dataset_groups': DATASET_GROUPS,
    'availability': availability_df.to_dict(orient='records') if 'availability_df' in globals() else [],
    'summary': summary_df.to_dict(orient='records') if 'summary_df' in globals() else [],
    'recommendations': recommendations_df.to_dict(orient='records') if 'recommendations_df' in globals() else [],
    'output_files': [
        'top_level_inventory.csv',
        'detected_dataset_candidates.csv',
        'dataset_availability_checklist.csv',
        'full_file_inventory.csv',
        'dataset_summary.csv',
        'extension_summary.csv',
        'split_summary.csv',
        'role_summary.csv',
        'folder_structure_analysis.csv',
        'pairing_check.csv',
        'dataset_recommendations.csv',
    ]
}

json_path = OUTPUT_DIR / 'dataset_audit_report.json'
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print('JSON report saved:', json_path)
print('\nMain reports are in:', OUTPUT_DIR.resolve())

JSON report saved: D:\Datasets\Satillate\_dataset_audit_reports\dataset_audit_report.json

Main reports are in: D:\Datasets\Satillate\_dataset_audit_reports


## How to use the results

After running all cells, check these files inside `_dataset_audit_reports`:

1. `dataset_availability_checklist.csv` — which expected datasets were detected.
2. `dataset_summary.csv` — size, file counts, split counts, and detected roles.
3. `folder_structure_analysis.csv` — likely image, label, split, and annotation folders.
4. `pairing_check.csv` — whether pre/post/label files seem to match by basename.
5. `dataset_recommendations.csv` — quick warnings and next steps.

For training, always visually inspect a few samples from each dataset using Cell 12 before building dataloaders.